: 

In [ ]:
import langchain

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

### Example 1: Simple LLM call with streaming

In [6]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

In [ ]:
model=init_chat_model("groq:llama-3.1-8b-instant")
model
#initialization of llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021EF81CC690>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021EF81CD090>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [ ]:
# create messages
messages=[
    SystemMessage("You are a helpful AI Assistant"),
    HumanMessage("What are the 2 benefits of langchain?")
]


In [11]:
# invoke the model
response=model.invoke(messages)
response

AIMessage(content='Langchain is an open-source artificial intelligence (AI) framework that integrates multiple AI models to create more powerful and flexible AI systems. Two benefits of Langchain include:\n\n1. **Multi-Model Integration**: Langchain allows developers to combine multiple AI models, such as language models (e.g., LLMs), computer vision models, and other specialized models, to create more comprehensive and robust AI systems. This integration enables these models to work together seamlessly, unlocking new possibilities for applications such as chatbots, virtual assistants, and more.\n\n2. **Dynamic Reasoning and Knowledge Graphs**: Langchain provides a system for dynamic reasoning and knowledge graph construction, which enables AI models to reason and interact with external knowledge sources in a more sophisticated and flexible way. This allows for the creation of AI systems that can engage in more natural and human-like conversations, and even learn and improve over time 

In [12]:
print(response.content)

Langchain is an open-source artificial intelligence (AI) framework that integrates multiple AI models to create more powerful and flexible AI systems. Two benefits of Langchain include:

1. **Multi-Model Integration**: Langchain allows developers to combine multiple AI models, such as language models (e.g., LLMs), computer vision models, and other specialized models, to create more comprehensive and robust AI systems. This integration enables these models to work together seamlessly, unlocking new possibilities for applications such as chatbots, virtual assistants, and more.

2. **Dynamic Reasoning and Knowledge Graphs**: Langchain provides a system for dynamic reasoning and knowledge graph construction, which enables AI models to reason and interact with external knowledge sources in a more sophisticated and flexible way. This allows for the creation of AI systems that can engage in more natural and human-like conversations, and even learn and improve over time based on the informatio

In [13]:
# another of invoking a model by direct humanMessage
model.invoke([HumanMessage("What is machine learning")])

AIMessage(content="Machine learning (ML) is a subset of artificial intelligence (AI) that involves training algorithms to learn from data, allowing them to make predictions, classify objects, and make decisions with minimal human intervention. The goal of machine learning is to enable machines to improve their performance on a task over time, without being explicitly programmed for that task.\n\nMachine learning involves the following key concepts:\n\n1. **Data**: Machine learning algorithms require large amounts of data to learn from. This data can be in the form of images, text, audio, or other types of data.\n2. **Training**: The algorithm is trained on the data, which involves adjusting the model's parameters to minimize the error between the predicted output and the actual output.\n3. **Model**: The trained algorithm is referred to as a model, which can be a statistical model, a decision tree, a neural network, or other type of model.\n4. **Prediction**: Once the model is trained,

In [ ]:
# Streaming Example
# to print response other than invoking model
for i in model.stream(messages):
    print(i.content, end=" ")

 Lang Chain  is  an  open -source  framework  for  building  large  language  models  ( LL Ms )  and  multim odal  AI  applications .  Some  of  its  benefits  include :

 1 .  ** Mod ular  Architecture **:  Lang Chain  allows  developers  to  build  complex  AI  applications  by  combining  different  modules  and  components ,  making  it  easier  to  integrate  and  customize  models ,  data ,  and  other  components .

 2 .  ** Eff icient  Model  Deployment **:  Lang Chain  provides  tools  and  libraries  that  enable  efficient  deployment  of  large  language  models ,  making  it  easier  to  integrate  models  into  production  environments ,  and  optimize  their  performance  for  specific  use  cases .   

### Dynamic Prompt Templates
if u need to give big instruction to llm, so there can be prompt template that can be used for structured way

In [26]:
from langchain_core.prompts import ChatPromptTemplate

# create translation app
translation_template=ChatPromptTemplate.from_messages([
    ("system","you are a professional translator, translate the following text- {text} from {source_language} to {target_language}. Maintain the tone and style"),     
    ("user","{text}")
])

# using template
prompt= translation_template.invoke({
    "source_language":"English",
    "target_language":"Spanish",
    "text":"Langchain makes building AI Applications incredibly easy!"
})

In [27]:
prompt

ChatPromptValue(messages=[SystemMessage(content='you are a professional translator, translate the following text- Langchain makes building AI Applications incredibly easy! from English to Spanish. Maintain the tone and style', additional_kwargs={}, response_metadata={}), HumanMessage(content='Langchain makes building AI Applications incredibly easy!', additional_kwargs={}, response_metadata={})])

In [28]:
translated_response= model.invoke(prompt)
print(translated_response.content)

Langchain hace que crear Aplicaciones de Inteligencia Artificial sea increíblemente fácil.


### Building Your First Chain

In [ ]:
from lanchain_core.output_parsers import StrOutParser
from lanchain_core.runnables import RunnablePassthrough, RunnableLambda
def create_story_chain():
    # template for story generation
    story_prompt=ChatPromptTemplate.from_messages(
        [
           ( "system","You are a creative story teller.Write a short and engaging story based on a given theme , character and settings"),
           ("user","Theme: {theme}\n Main character: {character}\n setting: {setting}")
        ]
    )

    #template for story analysis
    analysis_prompt= ChatPromptTemplate.from messages([
        ("system","You are a literacy critic. Analyse the follwing story and provide insights."),
        ("user","{story}")
    ])

    story_chain=(
        stroy_prompt|model|StrOutputParser
    )

    #create a function to pass the story to analysis
    def analyze_story(story_text):
        return{"story":story_text}
    
    analysis_chain=(
        story_chain
        |RunnableLambda(analyze_story)
        |analysis_prompt
        |model
        |strOutputParser()

    )

In [ ]:
result=chain.invoke({
    "theme":"artificial intelligence",
    "character":" a curious robot",
    "setting":"a futuritic city"
})
print("Story and Analysis")
print(result)

In [ ]:
story_chain